# Memory Tools for Agents
> Define and test tools for AI agents to interact with the SemanticMemory system.

## NBDev Default Export
Set the default export module name for nbdev.

In [ ]:
#| default_exp memorytools

## NBDev Hide
Import necessary nbdev functions like showdoc.

In [ ]:
#| hide
from nbdev.showdoc import *

## Import Required Libraries
Import `SemanticMemory` from the memory module, `Chat`, `toolloop`, `models` from `claudette`, and other necessary libraries like `json`.

In [ ]:
#| export
import json
from cogitarelink.memory import SemanticMemory # Assuming 00_memory.ipynb exports SemanticMemory
from claudette import Chat, toolloop, models, tool
from typing import Dict, List, Union

## Initialize Semantic Memory
Create an instance of the `SemanticMemory` class. We'll use this instance for our tools.

In [ ]:
#| export
# Global memory instance for tools to access
memory = SemanticMemory()

# Example: Seed memory with some initial data if needed for testing
# This could be loaded from a file or defined directly
initial_data = {
    "@context": {"@vocab": "http://schema.org/"},
    "@id": "http://example.org/initial_entity",
    "@type": "Thing",
    "name": "Initial Entity",
    "description": "An entity to start with."
}
memory.add_jsonld(initial_data)
print(f"Initialized memory: {memory}")

## Define Memory Query Tool
Create a function, annotated for tool use, that takes a query string, uses `memory.llm_query_translator` and `memory.retrieve_relevant_memory` to find relevant information, and returns it.

In [ ]:
#| export
@tool
def query_semantic_memory(natural_language_query: str, max_results: int = 5) -> List[Dict]:
    """
    Queries the semantic memory based on a natural language query.
    Translates the query into a structured format, retrieves relevant entities,
    and returns them in a compacted JSON-LD format.

    Args:
        natural_language_query: The user's query in natural language.
        max_results: The maximum number of relevant entities to return.

    Returns:
        A list of relevant entities found in the memory, compacted using a default context.
    """
    print(f"--- Tool: query_semantic_memory ---")
    print(f"Query: {natural_language_query}")
    # Step 1: Translate the natural language query to a structured query
    # Note: llm_query_translator might need an API key or specific setup
    # For now, we assume it works or use a simpler placeholder if needed
    try:
        structured_query = memory.llm_query_translator(natural_language_query)
        print(f"Structured Query: {structured_query}")
    except Exception as e:
        print(f"Error during query translation: {e}")
        # Fallback to basic keyword extraction if LLM fails
        structured_query = {"keywords": natural_language_query.split()}

    # Step 2: Retrieve relevant information from memory
    relevant_info = memory.retrieve_relevant_memory(structured_query, max_results=max_results)
    print(f"Retrieved {len(relevant_info)} entities.")
    return relevant_info

## Define Memory Storage Tool
Create a function, annotated for tool use, that takes JSON-LD data (as a string or dict) and adds it to the memory using `memory.add_jsonld` or `memory.add_named_graph`.

In [ ]:
#| export
@tool
def store_in_semantic_memory(data: Union[Dict, str], graph_id: str = None) -> str:
    """
    Stores JSON-LD data into the semantic memory.
    The data can be a single entity or a graph of entities.
    If a graph_id is provided, the data is added to that named graph.

    Args:
        data: The JSON-LD data to store, either as a Python dictionary or a JSON string.
        graph_id: Optional URI for a named graph to store the data in.

    Returns:
        A confirmation message indicating success or failure.
    """
    print(f"--- Tool: store_in_semantic_memory ---")
    print(f"Graph ID: {graph_id}")
    try:
        if isinstance(data, str): data = json.loads(data)

        if graph_id:
            print(f"Adding data to named graph: {graph_id}")
            memory.add_named_graph(data, graph_id)
            return f"Successfully added data to named graph '{graph_id}'."
        else:
            print("Adding data to default graph.")
            memory.add_jsonld_with_graph(data) # Use the graph-aware version
            # Determine the ID(s) added for a more informative message
            entity_id = data.get('@id', 'unknown') if isinstance(data, dict) else 'unknown'
            if isinstance(data, dict) and '@graph' in data: entity_id = f"{len(data['@graph'])} entities in graph"
            return f"Successfully added data (ID: {entity_id}) to the default graph."

    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}")
        return f"Error: Invalid JSON format provided. {e}"
    except Exception as e:
        print(f"Error storing data: {e}")
        return f"Error storing data in semantic memory: {e}"

## Define Context Listing Tool
Create a function, annotated for tool use, that calls `memory.list_contexts()` and returns the available contexts.

In [ ]:
#| export
@tool
def list_available_contexts() -> Dict:
    """
    Lists all available contexts registered in the semantic memory system.

    Returns:
        A dictionary where keys are context IDs and values contain details about each context.
    """
    print(f"--- Tool: list_available_contexts ---")
    contexts = memory.list_contexts()
    print(f"Found {len(contexts)} contexts.")
    return contexts

## Define Graph Listing Tool
Create a function, annotated for tool use, that calls `memory.list_named_graphs()` and returns the available named graphs.

In [ ]:
#| export
@tool
def list_available_graphs() -> Dict:
    """
    Lists all named graphs present in the semantic memory system.

    Returns:
        A dictionary where keys are graph IDs and values contain metadata about each graph (e.g., triple count).
    """
    print(f"--- Tool: list_available_graphs ---")
    graphs = memory.list_named_graphs()
    print(f"Found {len(graphs)} named graphs.")
    return graphs

## Integrate Tools with Claudette Agent
Instantiate a `Chat` object from `claudette`, passing the defined memory tools in the `tools` argument.

In [ ]:
# Select the Claude model
# model = models[0] # Haiku
model = models[1] # Sonnet 3.5
# model = models[2] # Opus

print(f"Using model: {model}")

# Create the agent (Chat instance) with the memory tools
memory_agent = Chat(model, tools=[
    query_semantic_memory,
    store_in_semantic_memory,
    list_available_contexts,
    list_available_graphs
])

print("Claudette agent initialized with memory tools.")

## Test Memory Tools with Agent
Create example prompts for the agent that require using the memory tools (e.g., 'Add this information about Alice...', 'What do you know about Alice?', 'List the available contexts.'). Use `chat.toolloop()` to run the agent.

In [ ]:
# Example data to add via the agent
alice_data = {
    "@context": {"@vocab": "http://schema.org/"},
    "@id": "http://example.org/person/alice",
    "@type": "Person",
    "name": "Alice Wonderland",
    "jobTitle": "Dreamer",
    "description": "Curious individual, prone to falling down rabbit holes."
}

# Convert dict to JSON string for the prompt
alice_json_string = json.dumps(alice_data)

# Define test prompts
prompts = [
    f"Please store the following information about Alice: ```json\n{alice_json_string}\n```",
    "What do you know about Alice Wonderland?",
    "List the available contexts in the memory.",
    "List the named graphs currently stored.",
    "Add the following data to a graph named 'http://example.org/graphs/people': ```json\n{{\"@context\":{{\"@vocab\":\"http://xmlns.com/foaf/0.1/\"}},\"@id\":\"http://example.org/person/bob\",\"@type\":\"Person\",\"name\":\"Bob The Builder\"}}\n```",
    "What information is stored in the graph 'http://example.org/graphs/people'?", # This might require a specific graph query tool if query_semantic_memory doesn't handle graphs well
    "Tell me about Bob The Builder."
]

# Run the agent with the prompts
results = []
for i, p in enumerate(prompts):
    print(f"\n--- Running Prompt {i+1} ---")
    print(f"Prompt: {p}")
    response = memory_agent.toolloop(p)
    print(f"\nAgent Response:\n{response}")
    results.append(response)
    # Clear chat history for next independent prompt if desired
    # memory_agent.clear()

In [ ]:
# Display the final state of the memory indices for inspection
print("\n--- Final Memory State ---")
print("by_id keys:", list(memory.indices['by_id'].keys()))
print("by_type keys:", list(memory.indices['by_type'].keys()))
print("Context Registry:", memory.list_contexts())
print("Named Graphs:", memory.list_named_graphs())

## NBDev Export
Export the notebook cells to the library.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()